# Customer Orders Analysis

This practice uses a small order-line dataset to answer three business questions:

1. Which products and categories are most popular?
2. Which customers are repeat or high-value customers?
3. Which products are commonly purchased together?

Each CSV row is one line item. An order can therefore have several rows. The analysis uses lists, tuples, dictionaries, sets, loops, conditionals, functions, and file handling from the Python standard library.

## 1. Load the dataset

The file schema is `order_id`, `customer_id`, `product`, `quantity`, `unit_price`, and `order_date`.

In [1]:
import csv
from pathlib import Path

data_path = Path("Datasets/customer_orders/customer_orders.csv")
if not data_path.exists():
    data_path = Path("../../Datasets/customer_orders/customer_orders.csv")

print(f"Reading: {data_path}")

Reading: Datasets\customer_orders\customer_orders.csv


In [2]:
def load_orders(file_path):
    """Read valid order lines and skip malformed rows with a clear message."""
    orders = []
    try:
        with file_path.open(newline="", encoding="utf-8") as order_file:
            reader = csv.DictReader(order_file)
            required_fields = {"order_id", "customer_id", "product", "quantity", "unit_price", "order_date"}
            if not required_fields.issubset(reader.fieldnames or []):
                raise ValueError("The CSV is missing one or more required columns.")
            for line_number, row in enumerate(reader, start=2):
                try:
                    quantity = int(row["quantity"])
                    unit_price = float(row["unit_price"])
                    if quantity <= 0 or unit_price < 0:
                        raise ValueError("quantity must be positive and unit_price cannot be negative")
                    orders.append({
                        "order_id": row["order_id"],
                        "customer_id": row["customer_id"],
                        "product": row["product"],
                        "quantity": quantity,
                        "unit_price": unit_price,
                        "order_date": row["order_date"],
                    })
                except (TypeError, ValueError) as error:
                    print(f"Skipping line {line_number}: {error}")
    except FileNotFoundError:
        print(f"Could not find {file_path}")
    return orders

orders = load_orders(data_path)
order_lines = [
    (item["order_id"], item["customer_id"], item["product"], item["quantity"], item["unit_price"])
    for item in orders
]

print(f"Loaded {len(orders)} order lines.")
print("First 5 tuple records:", order_lines[:5])

Loaded 116 order lines.
First 5 tuple records: [('1001', 'C001', 'Mechanical Keyboard', 1, 89.99), ('1001', 'C001', 'USB-C Cable', 1, 12.99), ('1001', 'C001', 'Desk Lamp', 1, 39.5), ('1002', 'C001', 'Standing Desk', 1, 289.0), ('1002', 'C001', 'Wireless Mouse', 1, 24.99)]


## 2. Classify products and calculate line revenue

A dictionary maps each known product to a category. The fallback category demonstrates how to handle a product that has not been mapped yet.

In [3]:
product_categories = {
    "Wireless Mouse": "Electronics",
    "USB-C Cable": "Electronics",
    "Mechanical Keyboard": "Electronics",
    "Noise-Canceling Headphones": "Electronics",
    "Standing Desk": "Office",
    "Desk Lamp": "Office",
    "Notebook": "Stationery",
    "Pen Set": "Stationery",
    "Water Bottle": "Fitness",
    "Running Shoes": "Fitness",
    "Yoga Mat": "Fitness",
}

def classify_product(product):
    return product_categories.get(product, "Unknown")

def line_revenue(order_line):
    return order_line["quantity"] * order_line["unit_price"]

for order_line in orders:
    order_line["category"] = classify_product(order_line["product"])
    order_line["revenue"] = line_revenue(order_line)

print(orders[0])

{'order_id': '1001', 'customer_id': 'C001', 'product': 'Mechanical Keyboard', 'quantity': 1, 'unit_price': 89.99, 'order_date': '01-02-2026', 'category': 'Electronics', 'revenue': 89.99}


## 3. Build summary dictionaries

The functions below use loops and dictionary updates to summarize products, categories, customers, and orders. Customer segments are based on total spend: High is at least $500, Medium is at least $250, and Regular is below $250.

In [4]:
def summarize_products(order_lines):
    summary = {}
    for item in order_lines:
        product = item["product"]
        if product not in summary:
            summary[product] = {"category": item["category"], "units": 0, "revenue": 0}
        summary[product]["units"] += item["quantity"]
        summary[product]["revenue"] += item["revenue"]
    return summary

def summarize_categories(order_lines):
    summary = {}
    for item in order_lines:
        category = item["category"]
        if category not in summary:
            summary[category] = {"units": 0, "revenue": 0}
        summary[category]["units"] += item["quantity"]
        summary[category]["revenue"] += item["revenue"]
    return summary

def customer_segment(total_spend):
    if total_spend >= 500:
        return "High"
    if total_spend >= 250:
        return "Medium"
    return "Regular"

def summarize_customers(order_lines):
    summary = {}
    for item in order_lines:
        customer = item["customer_id"]
        if customer not in summary:
            summary[customer] = {"orders": set(), "categories": set(), "spend": 0}
        summary[customer]["orders"].add(item["order_id"])
        summary[customer]["categories"].add(item["category"])
        summary[customer]["spend"] += item["revenue"]
    for customer_data in summary.values():
        customer_data["order_count"] = len(customer_data["orders"])
        customer_data["average_order_value"] = customer_data["spend"] / customer_data["order_count"]
        customer_data["segment"] = customer_segment(customer_data["spend"])
    return summary

def summarize_orders(order_lines):
    summary = {}
    for item in order_lines:
        order_id = item["order_id"]
        if order_id not in summary:
            summary[order_id] = {"customer_id": item["customer_id"], "products": set(), "revenue": 0}
        summary[order_id]["products"].add(item["product"])
        summary[order_id]["revenue"] += item["revenue"]
    return summary

product_summary = summarize_products(orders)
category_summary = summarize_categories(orders)
customer_summary = summarize_customers(orders)
order_summary = summarize_orders(orders)

product_ranking = sorted(product_summary.items(), key=lambda entry: (-entry[1]["units"], entry[0]))
category_ranking = sorted(category_summary.items(), key=lambda entry: (-entry[1]["revenue"], entry[0]))
customer_ranking = sorted(customer_summary.items(), key=lambda entry: (-entry[1]["spend"], entry[0]))

print("Top products by units:")
for product, values in product_ranking[:5]:
    print(f"  {product}: {values['units']} units, ${values['revenue']:.2f}")
print("\nCategories by revenue:")
for category, values in category_ranking:
    print(f"  {category}: ${values['revenue']:.2f}")

Top products by units:
  Pen Set: 25 units, $249.75
  Notebook: 24 units, $156.00
  Desk Lamp: 22 units, $869.00
  Running Shoes: 10 units, $950.00
  Water Bottle: 10 units, $180.00

Categories by revenue:
  Office: $2314.00
  Electronics: $1796.77
  Fitness: $1450.00
  Stationery: $405.75


## 4. Explore purchasing patterns with sets

Sets make unique values and relationships easy to inspect. Product pairs are counted only when both products occur in the same order.

In [5]:
unique_customers = {item["customer_id"] for item in orders}
unique_products = {item["product"] for item in orders}
unique_categories = {item["category"] for item in orders}
repeat_customers = {customer for customer, values in customer_summary.items() if values["order_count"] > 1}
high_value_customers = {customer for customer, values in customer_summary.items() if values["segment"] == "High"}

pair_counts = {}
for order in order_summary.values():
    products = sorted(order["products"])
    for first_index in range(len(products)):
        for second_index in range(first_index + 1, len(products)):
            pair = (products[first_index], products[second_index])
            pair_counts[pair] = pair_counts.get(pair, 0) + 1

pair_ranking = sorted(pair_counts.items(), key=lambda entry: (-entry[1], entry[0]))
all_electronics_customers = {customer for customer, values in customer_summary.items() if values["categories"] == {"Electronics"}}
fitness_customers = {customer for customer, values in customer_summary.items() if "Fitness" in values["categories"]}

print(f"Unique customers: {len(unique_customers)}")
print(f"Unique products: {len(unique_products)}")
print(f"Unique categories: {len(unique_categories)}")
print("Repeat customers:", sorted(repeat_customers))
print("High-value customers:", sorted(high_value_customers))
print("Customers who purchased Fitness products:", sorted(fitness_customers))
if pair_ranking:

    print("Products purchased together most often:", pair_ranking[0])    
else:
    print("Products purchased together most often: None (no multi-item orders in this dataset)")

Unique customers: 15
Unique products: 11
Unique categories: 4
Repeat customers: ['C001', 'C002', 'C003', 'C004', 'C005', 'C006', 'C007', 'C008', 'C009', 'C010', 'C011', 'C012', 'C014']
High-value customers: ['C001', 'C004', 'C005', 'C007', 'C010']
Customers who purchased Fitness products: ['C001', 'C002', 'C003', 'C004', 'C005', 'C006', 'C007', 'C008', 'C009', 'C010', 'C011', 'C012']
Products purchased together most often: (('Desk Lamp', 'USB-C Cable'), 8)


## 5. Business insights

The following output turns the summaries into short, decision-ready observations.

In [6]:
total_revenue = sum(item["revenue"] for item in orders)
average_order_value = total_revenue / len(order_summary)
most_popular_product = product_ranking[0]
highest_revenue_category = category_ranking[0]
top_customer = customer_ranking[0]

print(f"Total revenue: ${total_revenue:.2f}")
print(f"Number of orders: {len(order_summary)}")
print(f"Average order value: ${average_order_value:.2f}")
print(f"Most popular product: {most_popular_product[0]} ({most_popular_product[1]['units']} units)")
print(f"Highest-revenue category: {highest_revenue_category[0]} (${highest_revenue_category[1]['revenue']:.2f})")
print(f"Top customer by spend: {top_customer[0]} (${top_customer[1]['spend']:.2f}, {top_customer[1]['segment']})")
print(f"Repeat-customer count: {len(repeat_customers)}")
if pair_ranking:
    print(f"Strongest co-purchase pair: {pair_ranking[0][0]} ({pair_ranking[0][1]} orders)")
else:
    print("Strongest co-purchase pair: None (no multi-item orders found)")
print("Category behavior: customers can purchase across categories; comparing each customer's category set reveals cross-selling opportunities.")

Total revenue: $5966.52
Number of orders: 68
Average order value: $87.74
Most popular product: Pen Set (25 units)
Highest-revenue category: Office ($2314.00)
Top customer by spend: C004 ($1183.45, High)
Repeat-customer count: 13
Strongest co-purchase pair: ('Desk Lamp', 'USB-C Cable') (8 orders)
Category behavior: customers can purchase across categories; comparing each customer's category set reveals cross-selling opportunities.


## 6. Validate the analysis

Assertions check the main assumptions and ensure that the aggregate totals reconcile with the source rows.

In [7]:
assert len(orders) >= 100
# assert len(unique_customers) == 15
# assert len(unique_products) == 11
assert len(unique_categories) == 4
assert all(item["category"] != "Unknown" for item in orders)
assert all(item["revenue"] >= 0 for item in orders)
assert round(sum(item["revenue"] for item in orders), 2) == round(sum(values["revenue"] for values in category_summary.values()), 2)
assert round(sum(values["revenue"] for values in order_summary.values()), 2) == round(total_revenue, 2)
assert all(customer_summary[customer]["order_count"] > 1 for customer in repeat_customers)
assert not pair_ranking or pair_ranking[0][1] >= 1

print("All validation checks passed for the expanded dataset.")

All validation checks passed for the expanded dataset.


In [8]:
customer_totals = {}
category_totals = {}
customer_categories = {}
unique_products = set()

for item in orders:
    customer = item["customer_id"]
    category = item["category"]
    customer_totals[customer] = customer_totals.get(customer, 0) + item["revenue"]
    category_totals[category] = category_totals.get(category, 0) + item["revenue"]
    customer_categories.setdefault(customer, set()).add(category)
    unique_products.add(item["product"])

def classify_customer(total_spend):
    if total_spend > 100:
        return "High-value buyer"
    if total_spend >= 50:
        return "Moderate buyer"
    return "Low-value buyer"

print("Customer spending summary:")
for customer in sorted(customer_totals):
    total = customer_totals[customer]
    print(f"{customer}: ${total:.2f} -> {classify_customer(total)}")

print("\nCategory revenue summary:")
for category, revenue in sorted(category_totals.items(), key=lambda item: (-item[1], item[0])):
    print(f"{category}: ${revenue:.2f}")

print("\nUnique products:", sorted(unique_products))
print("Customers buying across multiple categories:")
for customer, categories in sorted(customer_categories.items()):
    if len(categories) > 1:
        print(f"{customer}: {sorted(categories)}")

print("\nTop 3 highest-spending customers:")
for customer, total in sorted(customer_totals.items(), key=lambda item: (-item[1], item[0]))[:3]:
    print(f"{customer}: ${total:.2f} -> {classify_customer(total)}")

print("\nBusiness insights:")
highest_category = max(category_totals.items(), key=lambda item: item[1])
top_customer = max(customer_totals.items(), key=lambda item: item[1])
most_common_product = max(
    ((product, sum(1 for item in orders if item["product"] == product)) for product in unique_products),
    key=lambda item: item[1],
)
print(f"Highest revenue category: {highest_category[0]} (${highest_category[1]:.2f})")
print(f"Top customer by spend: {top_customer[0]} (${top_customer[1]:.2f})")
print(f"Most frequently purchased product: {most_common_product[0]} ({most_common_product[1]} orders)")
print("Overall conclusion: electronics and office categories drive the strongest revenue, while repeat buyers across multiple categories indicate good cross-selling potential.")

Customer spending summary:
C001: $932.94 -> High-value buyer
C002: $259.98 -> High-value buyer
C003: $99.47 -> Moderate buyer
C004: $1183.45 -> High-value buyer
C005: $512.98 -> High-value buyer
C006: $158.98 -> High-value buyer
C007: $1018.46 -> High-value buyer
C008: $349.89 -> High-value buyer
C009: $82.98 -> Moderate buyer
C010: $956.96 -> High-value buyer
C011: $287.97 -> High-value buyer
C012: $82.98 -> Moderate buyer
C013: $9.99 -> Low-value buyer
C014: $19.50 -> Low-value buyer
C015: $9.99 -> Low-value buyer

Category revenue summary:
Office: $2314.00
Electronics: $1796.77
Fitness: $1450.00
Stationery: $405.75

Unique products: ['Desk Lamp', 'Mechanical Keyboard', 'Noise-Canceling Headphones', 'Notebook', 'Pen Set', 'Running Shoes', 'Standing Desk', 'USB-C Cable', 'Water Bottle', 'Wireless Mouse', 'Yoga Mat']
Customers buying across multiple categories:
C001: ['Electronics', 'Fitness', 'Office']
C002: ['Electronics', 'Fitness', 'Office', 'Stationery']
C003: ['Fitness', 'Station